In [11]:
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langgraph.types import interrupt, Command

In [41]:
class State(TypedDict):
    """The graph state."""
    story: str
    counter: int
    test: list[int]

In [ ]:
from langgraph.types import interrupt

def start_node(state: State):
    state["test"] = [0]
    state["counter"] = 0
    return state

def human_node(state: State):
    """Human node with validation."""
    state["test"].append(1)
    state["counter"] += 1

    question = "what is your name ?"

    answer = interrupt(question)

    state["test"].append(2)

    state["counter"] += 1


    state["story"] = answer

    return state

In [61]:
builder = StateGraph(State)
builder.add_node("start_node", start_node)  
builder.add_node("human_node", human_node)
builder.add_edge(START, "start_node")
builder.add_edge("start_node", "human_node")
builder.add_edge("human_node", END)
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)





In [62]:
config = {"configurable": {"thread_id": "1"}}
result = graph.invoke({}, config=config)
print(result)

pankaj is testing the interrupt 1
{'counter': 0, 'test': [0, 1], '__interrupt__': [Interrupt(value='what is your name ?', resumable=True, ns=['human_node:6e297eff-c7f0-a128-afee-6af1b3b00f3c'])]}


In [63]:
result

{'counter': 0,
 'test': [0, 1],
 '__interrupt__': [Interrupt(value='what is your name ?', resumable=True, ns=['human_node:6e297eff-c7f0-a128-afee-6af1b3b00f3c'])]}

In [64]:
result2 = graph.invoke(Command(resume="Rahul"), config=config)

pankaj is testing the interrupt 1


In [65]:
result2

{'story': 'Rahul', 'counter': 2, 'test': [0, 1, 2]}